# OLAF : creating a simple pipeline demo

In this demo, we create a simple pipeline using components from the OLAF library. The corpus is composed of basic sentences. We want to extract concepts and relations from it.

In [3]:

import os
import dill
import spacy
import pickle




# Import all necessary items from the olaf package
from olaf import Pipeline
from olaf.pipeline.pipeline_component.term_extraction import (
    POSTermExtraction,
    TFIDFTermExtraction,
    ManualCandidateTermExtraction
    )
from olaf.pipeline.pipeline_component.concept_relation_extraction import (
    CTsToConceptExtraction,
    CTsToRelationExtraction,
    SynonymRelationExtraction,
    SynonymConceptExtraction,
    AgglomerativeClusteringRelationExtraction,
    AgglomerativeClusteringConceptExtraction,
    LLMBasedConceptExtraction,
    LLMBasedRelationExtraction
)
from olaf.pipeline.pipeline_component.axiom_extraction.owl_axiom_extraction import OWLAxiomExtraction
from olaf.data_container.knowledge_representation_schema import KnowledgeRepresentation
from olaf.repository.serialiser import BaseOWLSerialiser
from olaf.repository.corpus_loader.text_corpus_loader import TextCorpusLoader
from olaf.data_container import CandidateTerm, Relation, Concept


/home/talibe/Bureau/Stage Insa/OLAF Research/olaf/env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [1]:
#ajouter par NFD
!python -m spacy download en_core_web_lg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 587.7/587.7 MB 10.6 MB/s eta 0:00:0000:0100:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')


In [5]:
# Load the spacy language model according to the corpus
spacy_model = spacy.load("en_core_web_lg")

In [35]:
# Initialise the corpus (for this example text version)
corpus = TextCorpusLoader(
    corpus_path="../data/metal/Casting_defect.txt",
)

In [36]:
bad_concept_pos = ["VERB","ADV","ADP","CCONJ","DET", "INTJ", "NUM","PRON", "PART", "SCONJ"]
bad_relation_pos = ["NOUN","ADV","ADP","CCONJ","DET", "INTJ", "NUM","PRON", "PART", "SCONJ"]
def get_bad_pos (bad_pos):
    def candidates_post_processing(candidates: set[CandidateTerm]) -> set[CandidateTerm]:
        
        list_label = []
        new_candidates = set()
        for candidate in candidates:
            keep = True
            if len(candidate.corpus_occurrences) > 0:
                for co in candidate.corpus_occurrences:
                    for token in co:
                        if (token.is_punct or token.is_stop or token.pos in bad_pos):
                            keep = False
                            break
                # print("good candidate: ", candidate.label)
            else:
                keep = False
                
            if keep and candidate.label not in list_label:
                new_candidates.add(candidate)
                list_label.append(candidate.label)
        return new_candidates
    return candidates_post_processing

In [ ]:
# relation candidates extraction

relation_pos = ["VERB"]

tfidf_relation_term_extraction = TFIDFTermExtraction( # documentation à regarder
    max_term_token_length=3,
    candidate_term_threshold=0.04,
    cts_post_processing_functions=[get_bad_pos(bad_relation_pos)]
)
pos_relation_term_extraction = POSTermExtraction(pos_selection=relation_pos)

# relation  extraction


ct_relation_extraction = CTsToRelationExtraction(concept_max_distance=6)
synonym_relation_extraction =  SynonymRelationExtraction(concept_max_distance=6)
agglo_relation_extraction = AgglomerativeClusteringRelationExtraction(distance_threshold=.4)


[2025-05-13 03:40:30,547] [WARNING] [tfidf_term_extraction] [_check_parameters] [Selected token sequence document attribute not set by the user.
                By default the system will use the entire content of the document.]
[2025-05-13 03:40:30,549] [WARNING] [pos_term_extraction] [__init__] [No preprocessing function provided for spans. Using the default one.]
[2025-05-13 03:40:30,549] [WARNING] [pos_term_extraction] [_check_parameters] [POS term extraction token sequence attribute not set by the user.
               By default the system will use the entire content of the document.]
[2025-05-13 03:40:30,550] [WARNING] [agglomerative_clustering_relation_extraction] [_check_parameters] [No value given for embedding_model parameter, default will be set to all-mpnet-base-v2.]
[2025-05-13 03:40:30,551] [WARNING] [agglomerative_clustering_relation_extraction] [_check_parameters] [No value given for metric option, default will be set to cosine.]
[2025-05-13 03:40:30,551] [WARNING] [agg

In [9]:
# concept candidates extraction

concepts_pos = ["NOUN"]

tfidf_concept_term_extraction = TFIDFTermExtraction( # documentation à regarder
    max_term_token_length=2,
    candidate_term_threshold=0.04,
    cts_post_processing_functions= [get_bad_pos(bad_concept_pos)]
)
pos_concept_term_extraction = POSTermExtraction(pos_selection=concepts_pos)
# pos_concept_term_extraction = ManualCandidateTermExtraction()


ct_concept_extraction = CTsToConceptExtraction()
synonym_concept_extraction = SynonymConceptExtraction()
agglo_concept_extraction = AgglomerativeClusteringConceptExtraction(distance_threshold=.4)
# llm_concept_extraction = LLMBasedConceptExtraction() # ne pas utiliser pour le moment

[2025-05-13 03:40:30,559] [WARNING] [tfidf_term_extraction] [_check_parameters] [Selected token sequence document attribute not set by the user.
                By default the system will use the entire content of the document.]
[2025-05-13 03:40:30,560] [WARNING] [pos_term_extraction] [__init__] [No preprocessing function provided for spans. Using the default one.]
[2025-05-13 03:40:30,561] [WARNING] [pos_term_extraction] [_check_parameters] [POS term extraction token sequence attribute not set by the user.
               By default the system will use the entire content of the document.]
[2025-05-13 03:40:30,561] [WARNING] [agglomerative_clustering_concept_extraction] [_check_parameters] [No value given for embedding_model parameter, default will be set to all-mpnet-base-v2.]
[2025-05-13 03:40:30,561] [WARNING] [agglomerative_clustering_concept_extraction] [_check_parameters] [No value given for metric option, default will be set to cosine.]


In [10]:
def clean_relations(kr: KnowledgeRepresentation):
    """
    Clean the relations in the knowledge representation
    :param kr: KnowledgeRepresentation
    :return: None
    """
    relations_to_remove = []
    for relation in kr.relations:
        if relation.source_concept is None or relation.destination_concept is None:
            relations_to_remove.append(relation)
        elif relation.source_concept.label == relation.destination_concept.label:
            relations_to_remove.append(relation)
    for relation in relations_to_remove:
        kr.relations.remove(relation)

def serialize_pipeline(pipeline, file_path):
    """
    Serialize the pipeline to a file
    :param pipeline: Pipeline
    :param file_path: str
    :return: None
    """
    components = pipeline.pipeline_components
    with open(file_path, 'wb') as f:
        dill.dump(components, f)

def deserialize_pipeline(file_path):
    """
    Deserialize the pipeline from a file
    :param file_path: str
    :return: Pipeline
    """
    with open(file_path, 'rb') as f:
        components = dill.load(f)
    return Pipeline(
        spacy_model=spacy_model,
        pipeline_components=components,
    )


# Pipeline 1
     - POSTermExtraction
     - CTsToConceptExtraction
     - POSTermExtraction
     - CTsToRelationExtraction


In [11]:
pipeline_1 = Pipeline(
    spacy_model=spacy_model,
    pipeline_components=[
        pos_concept_term_extraction, 
        CTsToConceptExtraction(),
        pos_relation_term_extraction, 
        CTsToRelationExtraction()],
   corpus_loader=corpus
)

pipeline_1.run()

# Clean the relations
clean_relations(pipeline_1.kr)

[2025-05-13 03:40:30,572] [WARNING] [candidate_terms_to_relations] [_check_parameters] [No value given for concept_max_distance parameter, default will be set to 5.]


In [12]:
for concept in pipeline_1.kr.concepts:
	print(concept)

cavities
blowholes
crack
millimetre
blisters
option
pools
indentation
melt
porosity
cooling
diameter
steam
shrinkage
form
rattails
smoke
filters
qualities
vacuum
gasses
terminology
spots
buckles
holes
terms
chemical
environment
foundry
system
top
line
practices
discontinuity
specification
solubility
preparation
things
nucleation
indentations
occurrence
swirl
cleanliness
argon
skin
castings
point
pulldowns
x
conditions
homogeneity
carbon
-
gates
impurities
nature
ray
imperfections
reasons
dioxide
scanning
interruptions
macroporosity
reason
analysis
center
erosion
flux
fineness
misruns
continuity
microporosity
workpiece
shuts
tolerance
accuracy
ductility
contact
extremities
irregularity
entrainment
eye
ways
deficiencies
inclusions
discontinuities
shape
fatigue
temperatures
distance
misrun
types
formation
tension
swell
fronts
category
oxide
toughness
turbulence
films
spot
causes
bubbles
coherency
wall
details
lack
solutions
fraction
velocity
mould
structure
fluidity
tears
materials
partic

In [13]:
for relation in pipeline_1.kr.relations:
	if relation.source_concept is not None or relation.destination_concept is not None:
		print(relation.source_concept, "-",  relation, "-", relation.destination_concept)


temperatures - kept - turbulence
oxygen - removed - phosphorus
mould - eliminate - formation
porosity - present - surface
preparation - design - occurrence
type - avoided - cooling
filters - gating - swirl
oxygen - removed - copper
metal - reduces - vicinity
air - minimize - temperatures
inclusions - occurs - liquid
shrinkage - contain - gases
mould - prevents - type
practices - melt - mould
run - occurs - metal
swirl - formed - liquid
mould - design - occurrence
mould - measuring - distance
part - moulding - cope
fluidity - changing - composition
defects - result - causes
liquid - occurs - mould
ingredients - added - mixture
ways - measuring - fluidity
cavity - leaving - portion
penetration - known - veining
metal - known - veining
oxides - interact - silica
casting - have - moulds
inclusions - reduce - oxide
penetration - occurs - metal
shrinkage - solidifies - defects
smoke - casting - sand
material - eroded - linings
cooling - includes - qualities
discontinuities - known - imperfec

In [14]:
my_olaf_demo1_serialiser = BaseOWLSerialiser("http://olaf_demo_results.org/")
my_olaf_demo1_serialiser.build_graph(pipeline_1.kr)

my_olaf_demo1_serialiser.export_graph("../data/metal/metal_default_pipeline_1_results.owl")


# Serialize the pipeline
serialize_pipeline(pipeline_1, "../data/metal/pipeline_metal_1.dill")

# Pipeline 2
	- TFIDFTermExtraction
	- CTsToConceptExtraction
	- TFIDFTermExtraction
	- CTsToRelationExtraction

In [18]:
pipeline_2 = Pipeline(
    spacy_model=spacy_model,
    pipeline_components=[
        TFIDFTermExtraction(
            max_term_token_length=2,
            candidate_term_threshold=0.03,
            cts_post_processing_functions=[get_bad_pos(bad_concept_pos)]
        ),
        ct_concept_extraction,
        TFIDFTermExtraction(
            max_term_token_length=2,
            candidate_term_threshold=0.03,
            cts_post_processing_functions=[get_bad_pos(bad_relation_pos)]
        ),
        ct_relation_extraction],
    corpus_loader=corpus
)

pipeline_2.run()
clean_relations(pipeline_2.kr)
print(("Concepts length: ", len(pipeline_2.kr.concepts)))

[2025-05-13 03:40:31,985] [WARNING] [tfidf_term_extraction] [_check_parameters] [Selected token sequence document attribute not set by the user.
                By default the system will use the entire content of the document.]
[2025-05-13 03:40:31,986] [WARNING] [tfidf_term_extraction] [_check_parameters] [Selected token sequence document attribute not set by the user.
                By default the system will use the entire content of the document.]
[2025-05-13 03:40:32,172] [WARNING] [tfidf_term_extraction] [_get_corpus_occurrences] [No corpus occurrence found for candidate term 0.5 mm]
[2025-05-13 03:40:32,173] [WARNING] [tfidf_term_extraction] [_get_corpus_occurrences] [No corpus occurrence found for candidate term 0.02 in]
[2025-05-13 03:40:32,174] [WARNING] [tfidf_term_extraction] [_get_corpus_occurrences] [No corpus occurrence found for candidate term 0.04 in]


good candidate:  solubility gases
good candidate:  surface finish
good candidate:  using various
good candidate:  inside
good candidate:  finish .
good candidate:  various additives
good candidate:  ductility
good candidate:  temperature ,
good candidate:  by sand
good candidate:  additives in
good candidate:  and cold
good candidate:  particles which
good candidate:  sand mixture
good candidate:  decreases
good candidate:  too coarse
good candidate:  mixture .
good candidate:  misruns and
good candidate:  coarse ,
good candidate:  improperly
good candidate:  to remove
good candidate:  of mould
good candidate:  material and
good candidate:  remove
good candidate:  or pouring
good candidate:  can occur
good candidate:  properly rammed
good candidate:  temperatures that
good candidate:  if
good candidate:  high .
good candidate:  spots .
good candidate:  . an
good candidate:  residual
good candidate:  an alternative
good candidate:  poured
good candidate:  mould is
good candidate:  alter

[2025-05-13 03:40:32,230] [WARNING] [tfidf_term_extraction] [_get_corpus_occurrences] [No corpus occurrence found for candidate term 0.5 mm]
[2025-05-13 03:40:32,230] [WARNING] [tfidf_term_extraction] [_get_corpus_occurrences] [No corpus occurrence found for candidate term 0.02 in]
[2025-05-13 03:40:32,231] [WARNING] [tfidf_term_extraction] [_get_corpus_occurrences] [No corpus occurrence found for candidate term 0.04 in]


good candidate:  length
good candidate:  pipes and
good candidate:  melt ,
good candidate:  factors with
good candidate:  taking actions
good candidate:  and caved
good candidate:  , steam
good candidate:  actions to
good candidate:  important factors
good candidate:  surfaces .
good candidate:  steam or
good candidate:  to keep
good candidate:  be important
good candidate:  . pipes
good candidate:  or smoke
good candidate:  keep them
good candidate:  pipes form
good candidate:  smoke from
good candidate:  them to
good candidate:  viscosity of
good candidate:  form at
good candidate:  casting sand
good candidate:  the required
good candidate:  casting and
good candidate:  sand ,
good candidate:  required level
good candidate:  and burrow
good candidate:  other gasses
good candidate:  level .
good candidate:  and viscosity
good candidate:  maximum
good candidate:  burrow into
good candidate:  gasses from
good candidate:  reduce
good candidate:  , while
good candidate:  melt or
good cand

In [19]:
docs = spacy_model("bubbles within")
for toke in docs:
	print(toke.text, toke.pos_, toke.dep_, toke.lemma_)

bubbles VERB ROOT bubble
within ADP prep within


In [20]:
for concept in pipeline_2.kr.concepts:
  print(concept)

fracture
dirty
melt
wash
dies
mould material
type
excessive lubricant
oxygen
present
slowly
metal contamination
pressure
blowholes
jagged
common
fronts
defect usually
larger
pore
finished casting
available
source
dioxide
liquid materials
minimum section
trapped
level
special
inside
usually
temperature
overly high
compensate
reduces
lubricant
metal penetration
thin
compensates
interruptions
metal shrinkage
defect
contact
fast
pouring temperatures
dissolved
prevents contact
install ceramic
defects include
required
rejected
eye
gas porosity
unfilled
penetration
slag
ductility
carbon dioxide
gas
corrected
superheat
vicinity
carbonaceous material
fatigue
result
volume
thick
sand mixture
naked
keeping
happens
freezing temperature
ceramic filters
practical
rammed mould
abnormal
increased
oxides
silica sands
conditions
hydrogen formation
nitrogen
cast
toughness
kept
gas formation
seen
decreases
actions
encountered
introduce
induce
solidifies
feed
taking
fill
formed
burrow
streamlined
solidifie

In [21]:
for relation in pipeline_1.kr.relations:
  if relation.source_concept is not None or relation.destination_concept is not None:
    print(relation.source_concept, "-",  relation, "-", relation.destination_concept)


temperatures - kept - turbulence
oxygen - removed - phosphorus
mould - eliminate - formation
porosity - present - surface
preparation - design - occurrence
type - avoided - cooling
filters - gating - swirl
oxygen - removed - copper
metal - reduces - vicinity
air - minimize - temperatures
inclusions - occurs - liquid
shrinkage - contain - gases
mould - prevents - type
practices - melt - mould
run - occurs - metal
swirl - formed - liquid
mould - design - occurrence
mould - measuring - distance
part - moulding - cope
fluidity - changing - composition
defects - result - causes
liquid - occurs - mould
ingredients - added - mixture
ways - measuring - fluidity
cavity - leaving - portion
penetration - known - veining
metal - known - veining
oxides - interact - silica
casting - have - moulds
inclusions - reduce - oxide
penetration - occurs - metal
shrinkage - solidifies - defects
smoke - casting - sand
material - eroded - linings
cooling - includes - qualities
discontinuities - known - imperfec

In [22]:
# Save the pipeline


In [23]:
my_olaf_demo2_serialiser = BaseOWLSerialiser("http://olaf_demo_results.org/")
my_olaf_demo2_serialiser.build_graph(pipeline_2.kr)

my_olaf_demo2_serialiser.export_graph( "../data/metal/metal_default_pipeline_2_results.owl")

serialize_pipeline(pipeline_2, "../data/metal/pipeline_metal_2_fr.dill")

# Pipeline 3
	- TFIDFTermExtraction
	- AgglomerativeClusteringConceptExtraction
	- TFIDFTermExtraction
	- AgglomerativeClusteringRelationtExtraction

In [24]:
pipeline_3 = Pipeline(
    spacy_model=spacy_model,
    pipeline_components=[
        tfidf_concept_term_extraction,
        agglo_concept_extraction,
        tfidf_relation_term_extraction,
        agglo_relation_extraction],
    corpus_loader=corpus
)

pipeline_3.run()
clean_relations(pipeline_3.kr)

/home/talibe/Bureau/Stage Insa/OLAF Research/olaf/env/lib/python3.12/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
[2025-05-13 03:40:36,135] [WARNING] [tfidf_term_extraction] [_get_corpus_occurrences] [No corpus occurrence found for candidate term 0.5 mm]
[2025-05-13 03:40:36,135] [WARNING] [tfidf_term_extraction] [_get_corpus_occurrences] [No corpus occurrence found for candidate term 0.02 in]
[2025-05-13 03:40:36,136] [WARNING] [tfidf_term_extraction] [_get_corpus_occurrences] [No corpus occurrence found for candidate term 0.04 in]


good candidate:  kept low
good candidate:  fluid
good candidate:  low .
good candidate:  value
good candidate:  . turbulence
good candidate:  ranges
good candidate:  turbulence from
good candidate:  0.4
good candidate:  from pouring
good candidate:  0.8
good candidate:  pouring the
good candidate:  point at
good candidate:  metal into
good candidate:  at which
good candidate:  can introduce
good candidate:  which the
good candidate:  introduce gases
good candidate:  not flow
good candidate:  the moulds
good candidate:  flow is
good candidate:  moulds are
good candidate:  is called
good candidate:  often streamlined
good candidate:  called the
good candidate:  streamlined to
good candidate:  the coherency
good candidate:  minimize such
good candidate:  coherency point
good candidate:  such turbulence
good candidate:  point .
good candidate:  turbulence .
good candidate:  is difficult
good candidate:  other methods
good candidate:  to predict
good candidate:  methods include
good candida

/home/talibe/Bureau/Stage Insa/OLAF Research/olaf/env/lib/python3.12/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
[2025-05-13 03:40:38,398] [WARNING] [tfidf_term_extraction] [_get_corpus_occurrences] [No corpus occurrence found for candidate term to 0.5 mm]
[2025-05-13 03:40:38,399] [WARNING] [tfidf_term_extraction] [_get_corpus_occurrences] [No corpus occurrence found for candidate term 0.5 mm (]
[2025-05-13 03:40:38,399] [WARNING] [tfidf_term_extraction] [_get_corpus_occurrences] [No corpus occurrence found for candidate term to 0.02 in]
[2025-05-13 03:40:38,400] [WARNING] [tfidf_term_extraction] [_get_corpus_occurrences] [No corpus occurrence found for candidate term 0.02 in )]
[2025-05-13 03:40:38,400] [WARNING] [tfidf_term_extraction] [_get_corpus_occurrences] [No corpus occurrence found for candidate term ( 0.04 in]
[2025-05-13 03:40:38,400] [WARNING] [tfidf_term_e

good candidate:  composition of
good candidate:  improperly vented mould
good candidate:  they must be
good candidate:  because most
good candidate:  composition of the
good candidate:  vented mould cavities
good candidate:  must be eliminated
good candidate:  slag
good candidate:  mould cavities .
good candidate:  , so the
good candidate:  be eliminated .
good candidate:  ladle
good candidate:  impurities
good candidate:  eliminated . they
good candidate:  . for
good candidate:  concentration
good candidate:  otherwise
good candidate:  they are broken
good candidate:  or slag
good candidate:  more
good candidate:  are broken down
good candidate:  the concentration
good candidate:  . this type
good candidate:  broken down into
good candidate:  the gas
good candidate:  concentration of
good candidate:  as the
good candidate:  down into five
good candidate:  of inclusions
good candidate:  steel
good candidate:  into five main
good candidate:  the concentration of
good candidate:  , such


In [25]:
for concept in pipeline_3.kr.concepts:
  print(concept)

appearance
jagged
castings
rough spots
furnace
mould cavity
number
cope
caved surfaces
flushing
hot
use
burn
mixture
residual
skimmed
decreases
low melting
leaving
easy
strain
gives way
fluidity affects
physical
sound
sharp corners
center
cold shuts
hot cracking
main
proud
compound
details
metal penetrates
greater
remove oxygen
wearing away
deemed
runners
seen
swirl gates
interact
metal shrinkage
carbonaceous
defect usually
residual stresses
coarse
second
present
measuring
repaired
rejected
rough
interruptions
filters
minimising hot
linings
larger
preparation
specific case
cope drops
ultrasonic
detrimental
region
formed
temperatures
area surrounding
portion
usually
shrinkage cavity
point
introduce gases
general
gates
rough surface
e.g.
molten metal
standard
magnetic
gating system
metal fills
shrinkage porosity
inert
oxygen
fast
usually involves
order
intended
closely
eliminated
scanning
mould erosion
smoke
actions
type
dirty
workpiece
called blowholes
material flows
control
weak
high p

In [26]:
for relation in pipeline_1.kr.relations:
  if relation.source_concept is not None or relation.destination_concept is not None:
    print(relation.source_concept, "-",  relation, "-", relation.destination_concept)


temperatures - kept - turbulence
oxygen - removed - phosphorus
mould - eliminate - formation
porosity - present - surface
preparation - design - occurrence
type - avoided - cooling
filters - gating - swirl
oxygen - removed - copper
metal - reduces - vicinity
air - minimize - temperatures
inclusions - occurs - liquid
shrinkage - contain - gases
mould - prevents - type
practices - melt - mould
run - occurs - metal
swirl - formed - liquid
mould - design - occurrence
mould - measuring - distance
part - moulding - cope
fluidity - changing - composition
defects - result - causes
liquid - occurs - mould
ingredients - added - mixture
ways - measuring - fluidity
cavity - leaving - portion
penetration - known - veining
metal - known - veining
oxides - interact - silica
casting - have - moulds
inclusions - reduce - oxide
penetration - occurs - metal
shrinkage - solidifies - defects
smoke - casting - sand
material - eroded - linings
cooling - includes - qualities
discontinuities - known - imperfec

In [27]:
# The knowledge representation should be empty before running the pipeline
pipeline_3.kr

KnowledgeRepresentation(concepts={appearance, jagged, castings, rough spots, furnace, mould cavity, number, cope, caved surfaces, flushing, hot, use, burn, mixture, residual, skimmed, decreases, low melting, leaving, easy, strain, gives way, fluidity affects, physical, sound, sharp corners, center, cold shuts, hot cracking, main, proud, compound, details, metal penetrates, greater, remove oxygen, wearing away, deemed, runners, seen, swirl gates, interact, metal shrinkage, carbonaceous, defect usually, residual stresses, coarse, second, present, measuring, repaired, rejected, rough, interruptions, filters, minimising hot, linings, larger, preparation, specific case, cope drops, ultrasonic, detrimental, region, formed, temperatures, area surrounding, portion, usually, shrinkage cavity, point, introduce gases, general, gates, rough surface, e.g., molten metal, standard, magnetic, gating system, metal fills, shrinkage porosity, inert, oxygen, fast, usually involves, order, intended, closel

In [30]:
my_olaf_demo3_serialiser = BaseOWLSerialiser("http://olaf_demo_results.org/")
my_olaf_demo3_serialiser.build_graph(pipeline_3.kr)

my_olaf_demo3_serialiser.export_graph( "../data/metal/metal_default_pipeline_3_results.owl")

try:
# Save the pipeline
	serialize_pipeline(pipeline_3, "../data/metal/pipeline_metal_3_fr.dill")
except Exception as e:
	print("Error during serialization:", e)
	

Error during serialization: [E112] Pickling a span is not supported, because spans are only views of the parent Doc and can't exist on their own. A pickled span would always have to include its Doc and Vocab, which has practically no advantage over pickling the parent Doc directly. So instead of pickling the span, pickle the Doc it belongs to or use Span.as_doc to convert the span to a standalone Doc object.


# Pipeline 4
	- POSTermExtraction
	- AgglomerativeClusteringConceptExtraction
	- POSTermExtraction
	- AgglomerativeClusteringRelationtExtraction

In [31]:
pipeline_4 = Pipeline(
    spacy_model=spacy_model,
    pipeline_components=[
        pos_concept_term_extraction,
        agglo_concept_extraction,
        pos_relation_term_extraction,
        agglo_relation_extraction],
    corpus_loader=corpus
)

pipeline_4.run()
clean_relations(pipeline_4.kr)

In [32]:


for concept in pipeline_4.kr.concepts:
  print(concept)

argon
sulfides
skin
drops
films
residues
shuts
factors
-
interruptions
entrainment
linings
flux
viscosity
sand
shape
cope
additives
nitrides
smoke
details
specification
order
line
freezing
thickness
shrinkage
die
flow
reactions
pipes
surrounding
bubbles
range
formation
pools
cavity
phosphorus
irregularity
types
fronts
moulding
extremities
swirl
cooling
actions
finish
inclusions
area
option
mixture
portion
venting
materials
homogeneity
compensates
lubricant
silica
underneath
layer
ray
concentration
workpiece
rattails
blowholes
composition
penetration
pore
tears
terms
pulldowns
qualities
imperfections
level
number
forms
oxygen
turbulence
cleanliness
cracking
gating
cases
contact
scabs
center
causes
steam
face
failures
fluidity
shear
point
hydrogen
stresses
ingredients
millimetre
way
fraction
grease
deficiencies
strain
scanning
castings
diameter
others
methods
wall
runners
region
eye
pour
pressure
system
environment
sections
tolerance
aluminium
continuity
buckle
flushing
carbon
things
sol

In [33]:
for relation in pipeline_1.kr.relations:
  if relation.source_concept is not None or relation.destination_concept is not None:
    print(relation.source_concept, "-",  relation, "-", relation.destination_concept)


temperatures - kept - turbulence
oxygen - removed - phosphorus
mould - eliminate - formation
porosity - present - surface
preparation - design - occurrence
type - avoided - cooling
filters - gating - swirl
oxygen - removed - copper
metal - reduces - vicinity
air - minimize - temperatures
inclusions - occurs - liquid
shrinkage - contain - gases
mould - prevents - type
practices - melt - mould
run - occurs - metal
swirl - formed - liquid
mould - design - occurrence
mould - measuring - distance
part - moulding - cope
fluidity - changing - composition
defects - result - causes
liquid - occurs - mould
ingredients - added - mixture
ways - measuring - fluidity
cavity - leaving - portion
penetration - known - veining
metal - known - veining
oxides - interact - silica
casting - have - moulds
inclusions - reduce - oxide
penetration - occurs - metal
shrinkage - solidifies - defects
smoke - casting - sand
material - eroded - linings
cooling - includes - qualities
discontinuities - known - imperfec

In [34]:
my_olaf_demo4_serialiser = BaseOWLSerialiser("http://olaf_demo_results.org/")
my_olaf_demo4_serialiser.build_graph(pipeline_4.kr)

my_olaf_demo4_serialiser.export_graph( "../data/metal/metal_default_pipeline_4_results.owl")

try:	
# Save the pipeline
	serialize_pipeline(pipeline_4, "../data/metal/pipeline_metal_4_fr.dill")
except Exception as e:
	print("Error during serialization:", e)

Error during serialization: [E112] Pickling a span is not supported, because spans are only views of the parent Doc and can't exist on their own. A pickled span would always have to include its Doc and Vocab, which has practically no advantage over pickling the parent Doc directly. So instead of pickling the span, pickle the Doc it belongs to or use Span.as_doc to convert the span to a standalone Doc object.
